# Week 6 — Four ways to answer from the course graph

Week 5 ended with a question answered from the graph, but the search, the traversal and
the starting point were written for that one question. This session builds a system that
takes **only a question**. You build the four retrievers of the lecture, a router that
chooses among them, and you score every one of them on one shared **test set**: five
questions about the course, each with the answer its documents state.

| Retriever | What it returns |
|---|---|
| `vector` | the three passages nearest in meaning (Week 3) |
| `name` | the graph's facts about the entities the question names |
| `hybrid` | the nearest passages, then the facts about everything they mention |
| `cypher` | the rows of a query the model writes, run in a read-only transaction |

Every retriever hands its sources to the same answering step, Week 3's grounded
instruction, unchanged. So the comparison at the end is a comparison of retrieval alone.

This is a rehearsal of **Lab 2**, whose central requirement is the same measured
comparison on your own domain, with your own test set.

**Before you start:** Neo4j running (`docker compose up -d` from the repository root);
`LIGHTNING_API_KEY` and `GEMINI_API_KEY` in `.env`. The session makes about 40 short
generation calls on Lightning and about 15 embeddings on Gemini. Lightning rate-limits
after roughly 28 calls in quick succession; the code then waits and retries by itself,
and prints that it is doing so.

In [1]:
# Run this notebook from any folder. Week 6's modules live in week6/; they reuse Week 2's
# answering step, Week 3's embeddings, Week 4's database helpers and Week 5's model.
import sys, pathlib
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "requirements.txt").exists()), pathlib.Path.cwd().parent)
for _d in ("common", "week2", "week3", "week4", "week5", "week6"):
    sys.path.insert(0, str(_root / _d))
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")

---
## Setup — the course graph, ready for retrieval

`week6/build_graph.py` rebuilds the graph in one step. It empties the database and loads
Week 5's reference extraction, with a few more aliases and settled types from
`week6/cleaning.py`, so that fewer names are split or mistyped. It copies Week 3's chunk
vectors onto the chunks, and adds the label `Entity` to every Topic, Tool and Deliverable,
a second label on the same node, so that one pattern reaches all three.

Last, it checks that the stored vectors come from the embedding model your client uses,
and records the model on the graph. Two embedding models can give vectors of the same
length in unrelated spaces, so a search with the wrong one would return passages ranked
by noise, without any error. Every search checks the record first. One embedding call,
no generation.

In [2]:
import json
from embedding_client import EmbeddingClient
from graph_db import breakdown, connect, run
from kg_loader import read_chunks
import build_graph

driver = connect()
build_graph.build(driver)
chunks = read_chunks()
emb = EmbeddingClient()

detail = breakdown(driver)
print("nodes by label:        " + ", ".join(f"{k} {v}" for k, v in detail["labels"].items()))
print("relationships by type: " + ", ".join(f"{k} {v}" for k, v in detail["relationships"].items()))

372 entities (2 names still under two types, 18 relationships dropped); 183 chunk vectors from gemini-embedding-2
nodes by label:        Chunk 183, Deliverable 22, Document 12, Entity 372, Meta 1, Tool 77, Topic 273, Week 3
relationships by type: ASSESSES 19, BUILDS_ON 48, FOR_WEEK 11, HAS_CHUNK 183, MENTIONS 613, NEXT 171, USES 70


You should see 372 entities, of which Topic 273, Tool 77 and Deliverable 22, and 183 chunk
vectors from the client's model.

---
## Part 1 — Read the test set

A test set is written **before** any retriever runs, from the documents, not from the
graph. Each question has the answer the corpus states and the chunks that state it.

In [3]:
TEST_PATH = _root / "week6" / "test_set.json"
test_set = json.loads(TEST_PATH.read_text(encoding="utf-8"))["questions"]
for q in test_set:
    print(f"{q['id']:4} {q['question']}")
    print(f"     expected: {q['expected']}")
    print(f"     kind:     {q['kind']}\n")

Q1   Why does the course prefer cosine similarity to Euclidean distance?
     expected: Cosine similarity measures direction only and discards magnitude; an embedding's magnitude can vary with properties such as text length that have nothing to do with meaning.
     kind:     passage

Q2   Which parts of the course use Chroma?
     expected: The Week 3 retrieval pipeline: Chroma is the vector store into which chunks and their embeddings are ingested and from which they are searched; it runs inside the Python process and persists to ./chroma/.
     kind:     relationship across chunks

Q3   Which databases does the course use to store embeddings?
     expected: Both: Chroma, the vector store of Week 3 (persisted to ./chroma/), and Neo4j, which holds vector indexes natively and is installed in Week 5 for Lab 2. An answer naming only one is partly correct.
     kind:     relationship across chunks: the answers' names are not in the question

Q4   How many of the course's documents mention

**How an answer is scored.** Give each answer one verdict:

| Verdict | Meaning |
|---|---|
| `correct` | every expected point, and nothing contradicting the cited chunks |
| `partly` | some expected points, nothing contradicting |
| `wrong` | a claim the cited chunks contradict, or the wrong answer |
| `silent` | "not stated" when an answer exists |

For Q5 the only correct answer is *not stated*: the corpus does not answer it. Wrong and
silent are kept apart because they cost the user differently: silence costs a question, a
confident wrong answer costs trust.

---
## Part 2 — Vector RAG

Every retriever returns **sources**: passages, or facts that each name the chunks stating
them. Every retriever also reads the graph through `read` in `retrievers.py`: Week 4's
`run`, in a read-only transaction, so that no retriever can write to the graph. Part 5
shows why that matters. `answer` then puts them before the question with Week 3's grounded instruction and
asks the model, through Lightning. With no sources it makes no call and returns
*not stated*.

`run_test` runs one retriever on all five questions, prints each answer beside the expected
one, and saves the answers to `week6/results/lab_answers.json`. If the kernel restarts, or
a call fails, run the cell again: it continues where it stopped. Pass `fresh=True` to run
again after changing a retriever.

In [4]:
from llm_client import LLMClient
from retrievers import answer

llm = LLMClient(provider="lightning")
SAVED = _root / "week6" / "results" / "lab_answers.json"
SAVED.parent.mkdir(exist_ok=True)
ANSWERS = json.loads(SAVED.read_text(encoding="utf-8")) if SAVED.exists() else {}


def run_test(name, retrieve, fresh=False):
    """Answer every test-set question from `retrieve(question)`, saving as it goes."""
    for q in test_set:
        key = f"{name} {q['id']}"
        if fresh or key not in ANSWERS:
            sources = retrieve(q["question"])
            ANSWERS[key] = {"answer": answer(llm, q["question"], sources),
                            "sources": len(sources)}
            SAVED.write_text(json.dumps(ANSWERS, indent=1, ensure_ascii=False), encoding="utf-8")
        got = ANSWERS[key]
        print(f"{q['id']:4} expected: {q['expected']}")
        print(f"     {name} ({got['sources']} sources): {' '.join(got['answer'].split())}\n")

The first retriever is Week 3's, now searching the chunk vectors in Neo4j as at the end of
Week 4: the three nearest passages.
It fetches 50 and keeps 3, because Week 4 measured the index as approximate at small `k`.
Look at the sources it gives the model for one question first:

In [5]:
from retrievers import vector

for s in vector(driver, emb, "Which databases does the course use to store embeddings?"):
    print(s[:300], "...\n")

notes: ≈4 min. Two points to land, and the second is the one that changes a student's design. First: capability does not follow from a data type. MySQL Community can store a vector and parse one from a string, and offers no distance function, no index, and no nearest-neighbour query — so a project o ...

their own project rather than adopting the course's default unexamined: dedicated vector databases such as Chroma, Qdrant, Weaviate and Milvus; general-purpose databases that have acquired vector search, including PostgreSQL through `pgvector`, SQL Server 2025, MariaDB, DynamoDB and — the one that m ...

--- ## Notes for the syllabus **`gemini-embedding-001`, named in §6 Week 3's optional exercise, is verified current (2026-09-13, recorded in `CLAUDE.md`) and its choice over the newer `gemini-embedding-2` is deliberate, not an oversight worth flagging away.** Measurement found that `-001` produces d ...



In [6]:
run_test("vector", lambda question: vector(driver, emb, question))

Q1   expected: Cosine similarity measures direction only and discards magnitude; an embedding's magnitude can vary with properties such as text length that have nothing to do with meaning.
     vector (3 sources): The course uses cosine similarity because it discards magnitude and measures direction, since magnitude can vary with text length and not with meaning. [1] The direction is the trained signal, so cosine similarity aligns with what the model has learned, unlike Euclidean distance. [2]

Q2   expected: The Week 3 retrieval pipeline: Chroma is the vector store into which chunks and their embeddings are ingested and from which they are searched; it runs inside the Python process and persists to ./chroma/.
     vector (3 sources): - Chroma is used in the Week 3 practicals/labs, running inside your Python process with a local embedded store. [1] - The retrieval_lab.ipynb, Part 1 of Week 3 practicals, demonstrates chunking and retrieval using Chroma. [1]

Q3   expected: Both: Chroma,

**Exercise 1.** Score each answer: one verdict per question. For one wrong answer, read the
chunk it cites, for example
`[c["text"] for c in chunks if c["id"] == "week-03-lecture.md#16"]`. Is the answer
faithful to a passage that is beside the point?

In [7]:
VERDICTS = {
    "vector": {"Q1": "correct", "Q2": "correct", "Q3": "partly", "Q4": "wrong", "Q5": "correct"},
}

---
## Part 3 — Graph by name

This retriever looks for the graph's entity names in the question, using Week 5's
`kg_model.key()`, so that *embeddings* finds `Embedding` through the alias table. It then
returns every fact touching those entities, each as one sentence naming its chunks. It
needs no embedding and no model call to retrieve.

It runs two queries. The first, `ENTITY_NAMES`, fetches every entity name; the matching
against the question happens in Python. The second, `FACTS`, takes the names found as the
parameter `$names`. `cypher_text` writes the parameter into the query, so that you can
read it, or paste it into Neo4j Browser and see the same rows.

In [8]:
from retrievers import ENTITY_NAMES, FACTS, by_name, cypher_text, facts_about, named_entities

question = "Which parts of the course use Chroma?"
names = named_entities(driver, question)
print(ENTITY_NAMES, "\n")
print("entities named in the question:", names, "\n")
print(cypher_text(FACTS, names=names), "\n")
for f in facts_about(driver, names):
    print("  ", f)

MATCH (e:Entity) RETURN e.name AS name 

entities named in the question: ['Chroma'] 

UNWIND ["Chroma"] AS n
MATCH (e:Entity {name: n})-[r:BUILDS_ON|USES|ASSESSES]-()
RETURN DISTINCT startNode(r).name AS a, type(r) AS rel, endNode(r).name AS b,
       r.chunks AS chunks
ORDER BY a, b 

   Embedding uses Chroma. (stated in week-03-practical-plan.md#6)
   Ingest uses Chroma. (stated in week-03-practical-plan.md#6, week-03-practical.md#13)
   Retrieval augmented generation uses Chroma. (stated in week-03-practical.md#1)
   Retrieval pipeline uses Chroma. (stated in week-03-lecture-plan.md#6)
   Vector store uses Chroma. (stated in week-03-practical.md#1)
   dense retrieval uses Chroma. (stated in week-03-practical-plan.md#0)


In [9]:
run_test("name", lambda question: by_name(driver, question))

Q1   expected: Cosine similarity measures direction only and discards magnitude; an embedding's magnitude can vary with properties such as text length that have nothing to do with meaning.
     name (4 sources): not stated.

Q2   expected: The Week 3 retrieval pipeline: Chroma is the vector store into which chunks and their embeddings are ingested and from which they are searched; it runs inside the Python process and persists to ./chroma/.
     name (6 sources): Embedding, Ingest, Retrieval augmented generation, Retrieval pipeline, Vector store, and Dense retrieval all use Chroma. [1][2][3][4][5][6]

Q3   expected: Both: Chroma, the vector store of Week 3 (persisted to ./chroma/), and Neo4j, which holds vector indexes natively and is installed in Week 5 for Lab 2. An answer naming only one is partly correct.
     name (6 sources): Chroma is used as the database to store embeddings. [2] The embeddings are stored in a VectorStore. [3]

Q4   expected: Four: the Week 3 lecture, lecture pl

The query graph by name ran for each test question. No model call: the names come from
the question alone. A question that names no entity runs with an empty list, and returns
no facts.

In [ ]:
for q in test_set:
    names = named_entities(driver, q["question"])
    print(f"--- {q['id']}  names: {names}")
    print(cypher_text(FACTS, names=names), "\n")

**Exercise 2.** Score these answers too. Then compare the two columns of verdicts. Which
retriever is silent, and on which questions? Which is wrong while sounding certain?
Silence costs the user a question; a confident wrong answer costs trust.

In [10]:
VERDICTS["name"] = {"Q1": "wrong", "Q2": "correct", "Q3": "partly", "Q4": "wrong", "Q5": "correct"}

---
## Part 4 — Hybrid retrieval

A question can enter the graph three ways: through the entity nearest in meaning (entity
vectors), through the entity names it contains (graph by name), or through the chunks
nearest in meaning and the entities they mention. Measured during preparation on ten
questions, the right starting entity was found for 5 by entity vectors, 4 by names, and 8
by chunks; names and chunks together found all 10. A chunk carries context that an
entity's name does not. The hybrid retriever does that: the
three nearest passages, then the facts about every entity they mention. Both go to the
model, passages first.

Read `hybrid` in `week6/retrievers.py`; it is six lines. Then look at what it gives the
model for the question vector retrieval answered only in part:

In [11]:
from retrievers import hybrid

sources = hybrid(driver, emb, "Which databases does the course use to store embeddings?")
print(len(sources), "sources")
for i, s in enumerate(sources, 1):
    print(f"[{i}] {s[:160]}")

26 sources
[1] notes: ≈4 min. Two points to land, and the second is the one that changes a student's design. First: capability does not follow from a data type. MySQL Communit
[2] their own project rather than adopting the course's default unexamined: dedicated vector databases such as Chroma, Qdrant, Weaviate and Milvus; general-purpose 
[3] --- ## Notes for the syllabus **`gemini-embedding-001`, named in §6 Week 3's optional exercise, is verified current (2026-09-13, recorded in `CLAUDE.md`) and it
[4] Choosing a model uses gemini-embedding-001. (stated in week-03-lecture.md#8, week-03-practical.md#8)
[5] Choosing a model uses gemini-embedding-2. (stated in week-03-lecture.md#8)
[6] Cosine distance uses Neo4j. (stated in week-03-lecture.md#16)
[7] Embedding uses Chroma. (stated in week-03-practical-plan.md#6)
[8] Embedding uses gemini-embedding-001. (stated in week-03-practical-plan.md#6)
[9] Embedding model uses gemini-embedding-001. (stated in week-03-lecture.md#5)
[10] Embedding m

In [12]:
run_test("hybrid", lambda question: hybrid(driver, emb, question))

Q1   expected: Cosine similarity measures direction only and discards magnitude; an embedding's magnitude can vary with properties such as text length that have nothing to do with meaning.
     hybrid (18 sources): Cosine similarity discards magnitude and measures direction alone, which matters because an embedding’s magnitude can vary with text length and other properties unrelated to meaning. [1] Therefore it’s preferred over Euclidean distance for ranking, since Euclidean distance is sensitive to magnitude while cosine distance (one minus similarity) captures the angle between vectors. [2]

Q2   expected: The Week 3 retrieval pipeline: Chroma is the vector store into which chunks and their embeddings are ingested and from which they are searched; it runs inside the Python process and persists to ./chroma/.
     hybrid (6 sources): Chroma is used in Week 3 practical work, including the hands-on embedding and chunking workflow. [1] It is also used in the retrieval_lab.ipynb notebook, 

In [13]:
VERDICTS["hybrid"] = {"Q1": "correct", "Q2": "partly", "Q3": "correct", "Q4": "wrong", "Q5": "correct"}

---
## Part 5 — Text-to-Cypher

Here the model writes the query. It is given the graph's schema, generated from
`kg_model.py` so that it cannot drift from the graph, and returns one field, `query`,
through structured output:

In [14]:
from retrievers import WRITE, schema_text, text_to_cypher

print(schema_text())

Nodes and their properties:
  (:Document {name, kind})
  (:Chunk {id, text, source})
  (:Week {number})
  (:Topic {name})
  (:Tool {name})
  (:Deliverable {name})
Relationships:
  (:Document)-[:HAS_CHUNK]->(:Chunk)
  (:Chunk)-[:NEXT]->(:Chunk)
  (:Document)-[:FOR_WEEK]->(:Week)
  (:Chunk)-[:MENTIONS]->(:Topic|Tool|Deliverable)
  (:Topic)-[:BUILDS_ON]->(:Topic)
  (:Topic)-[:USES]->(:Tool)
  (:Deliverable)-[:ASSESSES]->(:Topic)
Every BUILDS_ON, USES, ASSESSES relationship has r.chunks: the ids of the chunks that state it. Chunk ids look like 'week-03-lecture.md#7'.


A query written by a model is text from a source that is not you. It could write to the
database, run for ever, or return thousands of rows. The query is not inspected before it
runs; the safeguards are in how it runs:

| Safeguard | Catches |
|---|---|
| read-only transaction, `read` | every write, including APOC's procedures |
| 5-second timeout, 50 rows | runaway queries, an oversized context |

The read-only transaction is not special to this retriever: all four read through it.
Here it matters most, because the query is not ours. Try a write through it, as a query
the model wrote might:

In [15]:
from retrievers import read

attempt = "CREATE (:Topic {name: 'written by the model'})"
try:
    read(driver, attempt)
except Exception as exc:
    print("database:", str(exc).splitlines()[0][:160])
print("nodes written:", run(driver, "MATCH (t:Topic {name: 'written by the model'}) RETURN count(t) AS n")[0]["n"])

database: {neo4j_code: Neo.ClientError.Statement.AccessMode} {message: Writing in read access mode not allowed. Attempted write to neo4j} {gql_status: 50N42} {gql_status_
nodes written: 0


The read-only transaction is the safeguard that cannot be talked round: the database
enforces it. `text_to_cypher` reads the same way, through `execute_read`, so that it can
also set the 5-second timeout. Neo4j Community has no user roles, so this is the strongest
guarantee available without Enterprise. Inspecting the query text instead, for writing
clauses, would only be as good as the patterns written to find them; the database cannot
be talked round.

One more step concerns queries that are simply broken. A generated query sometimes uses a
variable that an earlier `WITH` dropped, and Neo4j rejects it with a syntax error. The
error names the fault exactly, so `text_to_cypher` sends the query back to the model once,
with the database's message, and prints `query rejected:` when it does. In the reference
runs of Q4, 2 of 16 queries were rejected, and every rejected query given back with its
error was corrected; a corrected query runs, which does not make it right.

The query is written by a larger model than the rest of the session uses: `gpt-5-mini`
(`WRITER_MODEL` in `retrievers.py`), while the answer is still written by `gpt-5-nano`, as
for the other retrievers. In preparation, nano's queries started from every document or
guessed an entity's label, and found the four documents that mention Chroma in none of
its runs; mini found them every time, at about twice the time per question.

Now one question, with the query the model wrote:

In [16]:
from retrievers import writer_for

print("query written by:", writer_for(llm).model, "| answer written by:", llm.model)
sources, query, problem = text_to_cypher(driver, llm, "How many of the course's documents mention Chroma?")
print(query, "\n")
print("problem:", problem)
for s in sources:
    print(s[:300])

query written by: openai/gpt-5-mini | answer written by: openai/gpt-5-nano
MATCH (t:Tool)<-[:MENTIONS]-(c:Chunk)<-[:HAS_CHUNK]-(d:Document)
WHERE toLower(t.name) CONTAINS 'chroma'
RETURN t.name AS tool, count(DISTINCT d) AS num_documents, collect(DISTINCT d.name) AS documents, collect(DISTINCT c.id) AS chunks 

problem: None
{'tool': 'Chroma', 'num_documents': 4, 'documents': ['week-03-practical-plan.md', 'week-03-lecture.md', 'week-03-practical.md', 'week-03-lecture-plan.md'], 'chunks': ['week-03-practical-plan.md#6', 'week-03-lecture.md#12', 'week-03-practical.md#13', 'week-03-lecture.md#15', 'week-03-practical-plan.m
{'tool': 'chroma browse', 'num_documents': 1, 'documents': ['week-03-practical-plan.md'], 'chunks': ['week-03-practical-plan.md#0', 'week-03-practical-plan.md#9']}


Run it on the test set. Each question costs two calls, one to write the query and one to
answer. The queries are kept in `CYPHER` for Exercise 4.

In [17]:
CYPHER = {}

def cypher_sources(question):
    sources, query, problem = text_to_cypher(driver, llm, question)
    CYPHER[question] = {"query": query, "problem": problem, "rows": len(sources)}
    return sources

run_test("cypher", cypher_sources)

Q1   expected: Cosine similarity measures direction only and discards magnitude; an embedding's magnitude can vary with properties such as text length that have nothing to do with meaning.
     cypher (0 sources): not stated (no sources)

Q2   expected: The Week 3 retrieval pipeline: Chroma is the vector store into which chunks and their embeddings are ingested and from which they are searched; it runs inside the Python process and persists to ./chroma/.
     cypher (2 sources): - Chroma is used in the retrieval-related parts of the course: retrieval pipeline, dense retrieval, ingest, embedding, retrieval augmented generation, and vector store. [1] - These topics appear in the Week 3 materials (lecture plan and practical plan). [1][2]

Q3   expected: Both: Chroma, the vector store of Week 3 (persisted to ./chroma/), and Neo4j, which holds vector indexes natively and is installed in Week 5 for Lab 2. An answer naming only one is partly correct.
     cypher (1 sources): It uses multimoda

In [ ]:
for q in test_set:
    got = CYPHER.get(q["question"])
    if got:
        print(f"--- {q['id']}  rows: {got['rows']}  problem: {got['problem']}")
        print(got["query"], "\n")

If `CYPHER` is empty, the answers came from the saved file; rerun the cell above with
`fresh=True` to see the queries.

**Exercise 4.** Score the answers. For each one that is wrong or silent, locate the fault
in its query:

- a query Neo4j **rejected** twice: `problem` is set and there are no rows;
- a **wrong label**: the model read *tool* in *tool calling* as the label `Tool`, but tool
  calling is a `Topic`;
- a **wrong direction**: `USES` points from a Topic to a Tool, `HAS_CHUNK` from a Document
  to a Chunk;
- an **invented property**: Neo4j runs the query anyway, with only a warning, and the
  property is empty in every row;
- rows that were **valid but beside the point**, or right rows the answer ignored.

None of these is a write, so the read-only transaction lets every one of them run. It
makes text-to-Cypher safe; it does not make it right.

In [18]:
VERDICTS["cypher"] = {"Q1": "wrong", "Q2": "correct", "Q3": "wrong", "Q4": "correct", "Q5": "correct"}

---
## Part 6 — The router

A user does not choose a retriever. One model call can: it reads the question and an
instruction, and returns the name of one retriever. Everything the router knows about the
retrievers is in that instruction, so this part builds it piece by piece. It needs three
things:

1. **What the material is.** The model has never seen this course. It must know that the
   material exists two ways, as passages and as a graph.
2. **What each retriever is for.** One line each: what the retriever returns, and the kind
   of question it suits.
3. **A closed choice.** Week 5's structured output again: the schema allows only the names
   of the four retrievers, so the model cannot choose one that does not exist.

In [19]:
from router import DATA, RETRIEVERS, Route, instruction, route

print("1. what the material is:\n  ", DATA, "\n")
print("2. what each retriever is for:")
for name, does in RETRIEVERS.items():
    print(f"   {name:7} {does}")
print("\n3. the closed choice:", Route.model_json_schema()["properties"]["retriever"]["enum"])

1. what the material is:
   You choose how to retrieve the information needed to answer a question about a university course. The course material is held two ways: as passages of text, and as a graph of topics, tools and deliverables extracted from it. Choose one retriever: 

2. what each retriever is for:
   vector  the passages nearest in meaning to the question. For a question answered by one passage: a definition, an explanation, a figure, an instruction.
   name    the graph's facts about an entity the question names exactly. For what a named topic, tool or deliverable builds on, uses or assesses.
   hybrid  the nearest passages, then the graph's facts about everything they mention. For a question about how things are connected when the entities are not named exactly.
   cypher  a database query written for the question. For counting, listing or comparing across the whole course.

3. the closed choice: ['vector', 'name', 'hybrid', 'cypher']


### How each line was written

Each description has two halves: **what the retriever returns**, which you saw in Parts 2
to 5, and **which questions it suits**, which follows from that.

| Retriever | Returns | So it suits |
|---|---|---|
| `vector` | the passages nearest in meaning | a question one passage answers: a definition, a figure, an instruction |
| `name` | facts about entities named in the question | what a *named* topic, tool or deliverable builds on, uses or assesses |
| `hybrid` | nearest passages, then facts about what they mention | how things are connected when the entities are *not* named exactly |
| `cypher` | rows of a query written for the question | counting, listing, comparing across the whole course |

Four rules shaped the wording:

- **Describe questions, not mechanisms.** The router sees only the question. "Embeds the
  question and searches an HNSW index" tells it nothing about which question to send; "a
  question one passage answers" does.
- **Separate the neighbours.** `name` and `hybrid` both return facts. What divides them is
  whether the question names its entity exactly, so both lines say so.
- **Never quote the test set.** An instruction that contains a test-set question, or its
  answer, is scored on questions it has already seen. The score would then say nothing
  about the next question a user asks. This is the same leak as testing a model on its
  training data.
- **Write the names once.** `Route` is built from the keys of `RETRIEVERS`, as Week 5 built
  extraction's entity types from `kg_model.py`, so a description and a choice cannot
  disagree.

`instruction()` joins the parts into the text the router receives:

In [20]:
print(instruction())

You choose how to retrieve the information needed to answer a question about a university course. The course material is held two ways: as passages of text, and as a graph of topics, tools and deliverables extracted from it. Choose one retriever:
- vector: the passages nearest in meaning to the question. For a question answered by one passage: a definition, an explanation, a figure, an instruction.
- name: the graph's facts about an entity the question names exactly. For what a named topic, tool or deliverable builds on, uses or assesses.
- hybrid: the nearest passages, then the graph's facts about everything they mention. For a question about how things are connected when the entities are not named exactly.
- cypher: a database query written for the question. For counting, listing or comparing across the whole course.


**Exercise 5.** You have read the instruction the router receives. Before it runs, predict
its choice for each question: which retriever will it pick, given only the question and
these four descriptions? Predict what the router *will* choose, not what it *should*.

In [21]:
ROUTER_PREDICTIONS = {"Q1": "vector", "Q2": "name", "Q3": "hybrid", "Q4": "cypher", "Q5": "vector"}

Now route every test-set question: five calls. Each row shows your prediction, the
router's choice, and the verdict you gave that retriever's answer to that question in
Parts 2 to 5. A choice is good when that verdict is `correct`.

In [22]:
ROUTES = {q["id"]: route(llm, q["question"]) for q in test_set}


def score_routes(routes):
    good = matched = 0
    print(f"{'':4} {'predicted':10} {'chose':8} {'its verdict':12} correct were")
    for q in test_set:
        chosen = routes[q["id"]]
        predicted = ROUTER_PREDICTIONS.get(q["id"]) or "-"
        verdict = VERDICTS.get(chosen, {}).get(q["id"]) or "unscored"
        correct = [r for r, v in VERDICTS.items() if v.get(q["id"]) == "correct"]
        good += verdict == "correct"
        matched += predicted == chosen
        print(f"{q['id']:4} {predicted:10} {chosen:8} {verdict:12} {correct or '-'}")
    print(f"\nyou predicted the router's choice for {matched} of {len(test_set)} questions")
    print(f"a retriever that answered correctly was chosen for {good} of {len(test_set)} questions")


score_routes(ROUTES)

     predicted  chose    its verdict  correct were
Q1   vector     vector   correct      ['vector', 'hybrid']
Q2   name       name     correct      ['vector', 'name', 'cypher']
Q3   hybrid     vector   partly       ['hybrid']
Q4   cypher     cypher   correct      ['cypher']
Q5   vector     vector   correct      ['vector', 'name', 'hybrid', 'cypher']

you predicted the router's choice for 4 of 5 questions
a retriever that answered correctly was chosen for 4 of 5 questions
